# 06 · Checkpoint: a recipe search engine with NumPy
Level L1 · Part 1 · builds on lessons 04, 05

## Goal
By the end of this lesson you can build a working semantic search over the 50 recipes, using only
NumPy and the saved embeddings: `search("seafood")` returns ranked recipes with scores, it runs
with no model and no network, and you can show with a printed test where it works and where it fails.

## The idea
Lessons 01–05 gave us every part of a search engine: recipes as vectors (02), a way to score
closeness (03), exact top-k (04), and a model that turns text into vectors (05). A checkpoint puts
the parts together behind one function, `search(text, k)`, the way a real product would call it.
A search engine is more than `knn`: it must keep vectors and recipes in step, turn the user's
words into a query vector, and say clearly when it cannot. We also test it, because "it printed
something" is not the same as "it works". Think of assembling flat-pack furniture: you already
have every piece, and now you find out whether they fit together.

## Picture

```mermaid
flowchart LR
    T["search('seafood', k=3)"] --> E["embed_query<br/>saved cache, or the model if loaded"]
    E --> K["knn(query, rows, k, 'dot')<br/>lesson 04"]
    D["recipes_50.csv + (50, 384) embeddings<br/>loaded and checked once"] --> W["which rows?<br/>all, or a where-mask"]
    W --> K
    K --> R["[(title, score), ...]"]
```

Everything in this picture is NumPy and pandas. The only model call is optional, inside `embed_query`.

## Requirements
Your engine is done when every box is ticked, with printed output to prove it:

- [ ] **R1 Load.** Read `recipes_50.csv` and the saved MiniLM embeddings; check that row *i* is recipe *i* and that every vector has length 1.
- [ ] **R2 Query vectors.** `embed_query(text)` returns a 384-number vector. With no model loaded it reads a saved cache; for a query not in the cache it raises a clear error.
- [ ] **R3 Search.** `search(text, k=3)` returns a list of `(title, score)`, best first.
- [ ] **R4 More like this.** `more_like(title, k=3)` returns the k recipes closest to a recipe, without that recipe itself.
- [ ] **R5 Test.** For a handful of queries with known good answers, print pass/fail for the top hit and a total.
- [ ] **R6 Filter.** `search(text, k, where=mask)` only considers rows where `mask` is True, so "quick vegetarian dinner" can be limited to vegetarian recipes ready in 30 minutes or less.

## Build it
Start from the two files lesson 05 saved and lesson 04's `knn` (copy it from lesson 05, step 7).
A useful order: R1, then R3 with a query from lesson 05's saved `queries`, then R2, R4, R5 and R6.
Write your engine in the cell below. Everything after it is the reference solution.

In [1]:
# your code here
# Suggested signatures:
#   embed_query(text) -> np.ndarray of shape (384,)
#   search(text, k=3, where=None) -> list[tuple[str, float]]
#   more_like(title, k=3) -> list[tuple[str, float]]

## Reference solution
**Try the build yourself first.** The cells below are one working answer, built requirement by
requirement. Compare your choices with them, not just your output.

### R1 · Load the recipes and their embeddings, and check them
Lesson 05 shipped both files. Two cheap checks at load time catch the worst bugs before any query
runs: titles in the same order (so row *i* really is recipe *i*) and unit length (so the cheaper
dot product equals cosine, lesson 03).

In [2]:
import numpy as np
import pandas as pd

recipes = pd.read_csv("../data/recipes_50.csv")
names = recipes["title"].tolist()
saved = np.load("../data/recipe_embeddings_minilm.npz")
doc_emb = saved["doc_emb"]
MODEL_NAME = str(saved["model"])

assert saved["names"].tolist() == names, "embeddings and CSV are out of step"
assert np.allclose(np.linalg.norm(doc_emb, axis=1), 1.0, atol=1e-5), "vectors are not unit length"
print("model:     ", MODEL_NAME)
print("recipes:   ", len(names), " embeddings:", doc_emb.shape, doc_emb.dtype)
print("columns:   ", recipes.columns.tolist())

model:      sentence-transformers/all-MiniLM-L6-v2
recipes:    50  embeddings: (50, 384) float32
columns:    ['id', 'title', 'description', 'cuisine', 'vegetarian', 'minutes']


Both checks pass silently, so the prints are reached. The `vegetarian` and `minutes` columns wait
until R6.

### The search core: lesson 04's `knn`, unchanged
Re-declared exactly as lessons 04 and 05 left it. Because the vectors have length 1, the engine
will call it with `metric="dot"`.

In [3]:
def top_k_smallest(scores, k):
    k = min(k, len(scores))
    part = np.argpartition(scores, k - 1)[:k]
    return part[np.argsort(scores[part], kind="stable")]

def knn(query, vectors, k, metric="euclidean"):
    q = np.asarray(query, dtype=np.float32)
    if metric == "euclidean":
        scores = np.linalg.norm(vectors - q, axis=1)
        rows = top_k_smallest(scores, k)                      # smaller = closer
    elif metric in ("dot", "cosine"):
        scores = vectors @ q
        if metric == "cosine":
            scores = scores / (np.linalg.norm(vectors, axis=1) * np.linalg.norm(q))
        rows = top_k_smallest(-scores, k)                     # bigger = closer
    else:
        raise ValueError(f"unknown metric {metric!r}")
    return rows, scores[rows]

print(knn(doc_emb[0], doc_emb, k=2, metric="dot"))

(array([0, 8]), array([1.       , 0.7118361], dtype=float32))


Searching with Tomato soup's own vector returns rows `[0, 8]`: row 0 is Tomato soup itself (score
1.0), row 8 is French onion soup (0.712). A recipe always finds itself first; keep that in mind for R4.

### R2 · Query vectors, with or without the model
A query must go through the same model as the recipes (lesson 05, mistake 2). With the model
available we embed a bank of queries once and save them next to the recipe embeddings. This cell
downloads the model, so it is tagged `needs-download`; `model = None` first means later cells
can tell whether it ran.

In [4]:
model = None
QUERY_BANK = [
    "something warm for a cold day", "a cold dessert for summer", "very spicy food",
    "something to eat in the morning", "quick vegetarian dinner", "Italian pasta", "seafood",
    "a chilled drink", "something with chickpeas", "a fancy dessert for guests", "a hot curry",
    "healthy salad", "comfort food for a rainy evening", "a dip for chips", "meat-free dinner",
]
print(len(QUERY_BANK), "queries in the bank")

15 queries in the bank


Fifteen queries: lesson 05's five plus ten new ones. Now embed and save them (skipped without
network; the course ships the file this cell writes).

In [5]:
import os, warnings
os.environ["TQDM_DISABLE"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
warnings.filterwarnings("ignore", message="IProgress")
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(MODEL_NAME, device="cpu")
np.savez("../data/query_embeddings_minilm.npz", model=MODEL_NAME,
         queries=QUERY_BANK, query_emb=model.encode_query(QUERY_BANK))
print("saved", len(QUERY_BANK), "query embeddings from", MODEL_NAME)

⏭  Skipped: this cell needs [download]. Run it yourself, or re-run with TUT_ENABLE=download.


Now `embed_query` itself: look in the saved cache first; if the text is not there, use the model if
it is loaded; otherwise raise a `KeyError` that says what to do. It also refuses a cache made by a
different model, the silent mistake from lesson 05.

In [6]:
cache = np.load("../data/query_embeddings_minilm.npz")
assert str(cache["model"]) == MODEL_NAME, f"query cache from {cache['model']}, recipes from {MODEL_NAME}"
query_cache = dict(zip(cache["queries"].tolist(), cache["query_emb"], strict=True))

def embed_query(text):
    if text in query_cache:
        return query_cache[text]
    if model is not None:
        return model.encode_query(text)
    raise KeyError(f"{text!r} is not in the saved query cache and the model is not loaded")

v = embed_query("seafood")
print("cached queries:", len(query_cache), " vector:", v.shape, v.dtype,
      " length:", round(float(np.linalg.norm(v)), 4))

cached queries: 15  vector: (384,) float32  length: 1.0


The query vector has the same shape and length as a recipe vector, so it can go straight into `knn`.

### R3 · `search(text, k)`
The whole engine in a few lines: text → vector → `knn` → titles. The `where` argument is for R6;
for now it is `None`, meaning "all rows". A small `show` prints results.

In [7]:
def search(text, k=3, where=None):
    allowed = np.arange(len(names)) if where is None else np.flatnonzero(where)
    rows, scores = knn(embed_query(text), doc_emb[allowed], k=k, metric="dot")
    return [(names[i], round(float(s), 3)) for i, s in zip(allowed[rows], scores, strict=True)]

def show(results, label):
    print(label)
    for rank, (title, score) in enumerate(results, start=1):
        print(f"  {rank}. {title:22} {score:.3f}")

for text in ["seafood", "Italian pasta", "something with chickpeas"]:
    show(search(text), f"search({text!r}):")

search('seafood'):
  1. Grilled salmon         0.502
  2. Sushi rolls            0.489
  3. Fish tacos             0.473
search('Italian pasta'):
  1. Spaghetti carbonara    0.706
  2. Pesto pasta            0.682
  3. Minestrone             0.633
search('something with chickpeas'):
  1. Falafel wrap           0.514
  2. Hummus                 0.506
  3. Quinoa salad           0.475


Three queries, three sensible pages: fish dishes, pasta, and the two chickpea dishes (Falafel wrap,
Hummus) first. No recipe contains the word "seafood"; Grilled salmon wins on meaning alone.

### R4 · More like this
The query is a recipe's own vector. Ask for k + 1 and drop the recipe itself (lesson 04, mistake 3).

In [8]:
def more_like(title, k=3):
    me = names.index(title)
    rows, scores = knn(doc_emb[me], doc_emb, k=k + 1, metric="dot")
    return [(names[i], round(float(s), 3)) for i, s in zip(rows, scores, strict=True) if i != me][:k]

for title in ["Greek salad", "Ramen", "Tiramisu"]:
    show(more_like(title), f"more_like({title!r}):")

more_like('Greek salad'):
  1. Caprese salad          0.696
  2. Caesar salad           0.684
  3. Quinoa salad           0.651
more_like('Ramen'):
  1. Chicken noodle soup    0.694
  2. Shrimp pad thai        0.607
  3. French onion soup      0.569
more_like('Tiramisu'):
  1. Pesto pasta            0.440
  2. Vegetable samosas      0.409
  3. Avocado toast          0.404


Greek salad's neighbours are the other salads (0.65–0.70) and Ramen's are noodle and broth dishes.
Tiramisu, lesson 05's odd one out, gets Pesto pasta at only 0.440: like `knn`, `more_like` always
answers, even when nothing is close (lesson 04, step 6). No query text is needed, so this works
offline for any of the 50 recipes.

### R5 · Test it
Write down, for each test query, which recipes would be a good *top* hit, then compare. For "quick
vegetarian dinner" any vegetarian recipe ready in 30 minutes or less counts.

In [9]:
quick_veg = recipes["vegetarian"] & (recipes["minutes"] <= 30)
tests = {
    "Italian pasta": {"Spaghetti carbonara", "Pesto pasta"},
    "seafood": {"Grilled salmon", "Sushi rolls", "Fish tacos", "Moules marinieres", "Shrimp pad thai"},
    "a hot curry": {"Vindaloo", "Thai green curry"},
    "something with chickpeas": {"Falafel wrap", "Hummus"},
    "a chilled drink": {"Mango lassi"},
    "something warm for a cold day": {"Chicken noodle soup", "Tomato soup", "Minestrone", "Beef stew",
                                      "Red lentil dal", "French onion soup", "Ramen"},
    "quick vegetarian dinner": set(recipes.loc[quick_veg, "title"]),
}
passed = 0
for text, good in tests.items():
    top = search(text, k=1)[0][0]
    passed += top in good
    print(f"{'PASS' if top in good else 'FAIL'}  {text:30} top hit: {top}")
print(f"\n{passed} of {len(tests)} passed")

PASS  Italian pasta                  top hit: Spaghetti carbonara
PASS  seafood                        top hit: Grilled salmon
PASS  a hot curry                    top hit: Vindaloo
PASS  something with chickpeas       top hit: Falafel wrap
PASS  a chilled drink                top hit: Mango lassi
PASS  something warm for a cold day  top hit: Chicken noodle soup
FAIL  quick vegetarian dinner        top hit: Beef stew

6 of 7 passed


Six of seven pass. The failure is real and important: "quick vegetarian dinner" returns Beef stew,
a 3-hour meat dish, exactly as lesson 05 saw. The model hears "dinner" and matches a hearty main;
it does not check facts like "vegetarian" or "quick". A test table like this is the seed of proper
evaluation (lesson 16).

### R6 · Filter on hard facts
Facts belong in columns, not in the embedding. A boolean mask over the 50 rows says which recipes
are allowed; `search` scores only those (`np.flatnonzero` turns the mask into row numbers) and
maps the answers back to the full table.

In [10]:
print("allowed rows:", int(quick_veg.sum()), "of", len(names))
show(search("quick vegetarian dinner"), "no filter:")
show(search("quick vegetarian dinner", where=quick_veg), "vegetarian and <= 30 minutes:")
hits = recipes.set_index("title").loc[[t for t, _ in search("quick vegetarian dinner", where=quick_veg)]]
print(hits[["vegetarian", "minutes"]].to_string())

allowed rows: 25 of 50
no filter:
  1. Beef stew              0.559
  2. Full English breakfast 0.515
  3. Guacamole              0.484
vegetarian and <= 30 minutes:
  1. Guacamole              0.484
  2. Mango salsa            0.455
  3. Spicy tofu stir-fry    0.444
                     vegetarian  minutes
title                                   
Guacamole                  True       10
Mango salsa                True       10
Spicy tofu stir-fry        True       20


Now every result is vegetarian and quick. The filter fixed the *facts*; the *ranking* is still the
model's: Guacamole and Mango salsa are dips, not dinners, because no description says "dinner".
Filtering inside the database, with indexes, is lesson 11; better ranking comes in Part 3.

## What just happened
- R1 loaded 50 recipes and a `(50, 384)` matrix and checked order and unit length before any query ran.
- R2's `embed_query` served 15 saved query vectors with no model, and would use the model when loaded.
- R3's `search` is lesson 04's `knn` plus bookkeeping: "seafood" found Grilled salmon without sharing a word.
- R4 found the salads next to Greek salad and the noodle dishes next to Ramen, dropping the recipe itself; Tiramisu's best match scored only 0.440.
- R5 printed 6 of 7 passing: semantic search is good at topic and bad at facts (Beef stew for "vegetarian").
- R6 restricted the search to 25 vegetarian, quick rows and every result obeyed, while the order still showed the model's limits.

## Common mistakes

**1. Using row numbers from the filtered matrix as if they were table rows.** `knn` on
`doc_emb[allowed]` returns positions *inside the smaller matrix*. Looking those up in `names`
picks the wrong recipes:

In [11]:
try:
    allowed = np.flatnonzero(quick_veg)
    rows, scores = knn(embed_query("quick vegetarian dinner"), doc_emb[allowed], k=3, metric="dot")
    wrong = [names[i] for i in rows]                         # BUG: rows index `allowed`, not `names`
    print(wrong)
    assert recipes.set_index("title").loc[wrong, "minutes"].le(30).all(), "a slow recipe got through the filter"
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

['Waldorf salad', 'Tomato soup', 'Strawberry cheesecake']
AssertionError: a slow recipe got through the filter


Positions in the 25-row matrix were looked up in the 50-row table, so the titles (Waldorf salad,
Tomato soup, Strawberry cheesecake) are unrelated recipes. Tomato soup takes 35 minutes, so it prints
`AssertionError: a slow recipe got through the filter`. Fix: map back first with `allowed[rows]`, as
`search` does.

In [12]:
print([names[i] for i in allowed[rows]])

['Guacamole', 'Mango salsa', 'Spicy tofu stir-fry']


**2. Filtering after the search.** Take the top 3, *then* drop non-vegetarian ones. It looks
equivalent, but the page comes back short:

In [13]:
try:
    top3 = search("quick vegetarian dinner", k=3)
    page = [(t, s) for t, s in top3 if t in tests["quick vegetarian dinner"]]
    print(page)
    assert len(page) == 3, f"asked for 3 results, got {len(page)}"
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

[('Guacamole', 0.484)]
AssertionError: asked for 3 results, got 1


It prints `AssertionError: asked for 3 results, got 1`: Beef stew and Full English breakfast were
removed and nothing replaced them. Fix: filter *before* ranking (the `where=` mask), which
always fills the page when enough rows qualify. Lesson 11 names these pre- and post-filtering.

**3. Expecting any text to work offline.** Without the model, only the 15 cached queries have
vectors. A new query cannot be answered:

In [14]:
try:
    search("party snacks to share")
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

KeyError: "'party snacks to share' is not in the saved query cache and the model is not loaded"


It prints `KeyError: "'party snacks to share' is not in the saved query cache and the model is not
loaded"` (with the model loaded, as after a `TUT_ENABLE=download` run, it is embedded instead).
Fix: add the text to `QUERY_BANK` and run the R2 download cell once to refresh the cache. Never
patch the gap with a vector from another model (lesson 05, mistake 2).

## Try it
1. **Quick desserts.** Search "a fancy dessert for guests" with no filter, then only among recipes
   ready in 30 minutes or less. Which top-3 recipes change?
2. **Know when to say nothing.** Add a `min_score` argument to `search` that drops results scoring
   below it. With `min_score=0.5`, which queries in `QUERY_BANK` return an empty page?

In [15]:
# your code here

## Key terms
- **search engine (semantic)** — a function from query text to ranked results: embed the query, score stored vectors, return the top-k with their scores.
- **query cache** — query embeddings computed once and saved, so the same queries can be answered later without the model.
- **filter (mask)** — a True/False per stored item from hard facts (vegetarian, minutes); only allowed items are ranked.
- **test query** — a query with known good answers, used to check that search results are right, not just printed.